### Step in depth antenna

A step in width microstrip antenna is a design variation where the radiating patch element has a non-uniform width that changes along its length. Instead of a simple rectangular patch, the conducting strip gradually narrows or widens at specific points.

In [1]:
import gmsh
import math
import os
import json

from palacetoolkit.viz import view_mesh
from palacetoolkit.geometry import extract_tag, xmin, xmax, ymin, ymax, zmin, zmax
from palacetoolkit.mesh import refine_near_surfaces, refine_surfaces

### Parameters:

- l1 : Ground plane length along x-axis, specified as a scalar in meters
- w1 : Ground plane width along y-axis, specified as a scalar in meters
- h : Patch height along z-axis, specified as a scalar in meters.

- strip_line_length : Notch length along x-axis, specified as a scalar in meters. 
- strip_lined_width_near_port: Notch width along x-axis near the port, specified as a scalar in meters. 
- strip_lined_width_far: Strip line width along y-axis far from the port, specified as a scalar in meters.
- air_height : Air box height along z-axis, specified as a scalar in meters.  
- air_margin : Air box margin along x and y axes, specified as a scalar in meters.
- freq  : Simulation frequency in GHz, specified as a scalar.
- filename : Output mesh filename, specified as a string.

In [2]:
l1: float = 0.06
w1: float = 0.06
strip_line_length: float = 0.06
strip_line_width_near_port: float = 0.001
strip_line_width_far: float = 0.003
h: float = 0.0013
air_height: float = 0.025    
air_margin: float = 0.025    
freq: float = 3.3
filename: str = "sw_antenna.msh"

### Initialize the model

In [3]:
gmsh.initialize()
gmsh.model.add("patch_antenna")
kernel = gmsh.model.occ

### Geometry generation


In [4]:
# Total domain bounds
total_xmin = -l1/2 - air_margin
total_xmax = l1/2 + air_margin
total_ymin = -w1/2 - air_margin
total_ymax = w1/2 + air_margin
total_zmax = h + air_height

substrate = kernel.addBox(-l1/2, -w1/2, 0, l1, w1, h)

ground_plane = kernel.addRectangle(-l1/2, -w1/2, 0, l1, w1)

strip_line_1 = kernel.addRectangle(-l1/2, -strip_line_width_near_port/2, h, strip_line_length/2, strip_line_width_near_port)
strip_line_2 = kernel.addRectangle(0, -strip_line_width_far/2, h, strip_line_length/2, strip_line_width_far)

top_conductor, _ = kernel.fuse(
    [(2, strip_line_1)], [(2, strip_line_2)],
    removeObject=True, removeTool=True
)
kernel.synchronize()


gap = 0
lumped_port = kernel.addRectangle(-l1/2 + gap, -strip_line_width_near_port/2, 0, h - gap, strip_line_width_near_port)
kernel.rotate([(2, lumped_port)], -l1/2, 0, 0, 0, 1, 0, -math.pi/2)
kernel.synchronize()

air_box = kernel.addBox(
    total_xmin, total_ymin, 0,
    total_xmax - total_xmin,
    total_ymax - total_ymin,
    total_zmax
)
kernel.synchronize()

all_tools = [(2, ground_plane)] + top_conductor + [(2, lumped_port)]

result, result_map = kernel.fragment(
    [(3, substrate), (3, air_box)], 
    all_tools
)
kernel.synchronize()
kernel.removeAllDuplicates()
kernel.synchronize()

Info    : Cannot bind existing OpenCASCADE surface 8 to second tag 9
Info    : Could not preserve tag of 2D object 9 (->8)


### Find the physical groups

In [5]:
# regions
all_2d = gmsh.model.getEntities(2)
all_3d = gmsh.model.getEntities(3)

tol = 1e-6

ground_tags = []
sides = []
patch_tags = []
port_tags = []
farfield_tags = []
substrate_tags = []
air_tags = []

# 2D surfaces
for dim, tag in all_2d:
    bbox = gmsh.model.getBoundingBox(dim, tag)
    xmin, ymin, zmin = bbox[0], bbox[1], bbox[2]
    xmax, ymax, zmax = bbox[3], bbox[4], bbox[5]
    
    # Ground plane
    if (math.isclose(zmin, 0, abs_tol=tol) and 
        math.isclose(zmax, 0, abs_tol=tol) and
        xmin >= -l1/2 - tol and xmax <= l1/2 + tol):
        ground_tags.append(tag)
        
    # Patch
    elif (math.isclose(zmin, h, abs_tol=tol) and 
            math.isclose(zmax, h, abs_tol=tol) and
            math.isclose(ymax, strip_line_width_far/2, abs_tol=tol)):
        patch_tags.append(tag)
        
    # Lumped port
    elif (math.isclose(xmin, -l1/2, abs_tol=tol) and 
            math.isclose(xmax, -l1/2, abs_tol=tol) and
            math.isclose(ymin, -strip_line_width_near_port/2, abs_tol=tol) and
            math.isclose(ymax, strip_line_width_near_port/2, abs_tol=tol) and
            zmin >= gap - tol and zmax <= h + tol):
        port_tags.append(tag)
        
    # Far-field (outer air box surfaces)
    elif (math.isclose(xmin, -l1/2 - air_margin, abs_tol=tol) or
            math.isclose(xmax, l1/2 + air_margin, abs_tol=tol) or
            math.isclose(ymin, -w1/2 - air_margin, abs_tol=tol) or
            math.isclose(ymax, w1/2 + air_margin, abs_tol=tol) or
            math.isclose(zmax, h + air_height, abs_tol=tol)):
        farfield_tags.append(tag)
    else:
        sides.append(tag)

# 3D volumes
for dim, tag in all_3d:
    bbox = gmsh.model.getBoundingBox(dim, tag)
    zmax = bbox[5]
    
    if math.isclose(zmax, h, abs_tol=tol):
        substrate_tags.append(tag)
    else:
        air_tags.append(tag)

# physical groups
pg_map = {
}

if substrate_tags:
    pg_map["substrate"] = gmsh.model.addPhysicalGroup(3, substrate_tags, name = "Substrate")
if air_tags:
    pg_map["air"] = gmsh.model.addPhysicalGroup(3, air_tags, name = "Air")
if ground_tags:
    pg_map["ground_plane"] = gmsh.model.addPhysicalGroup(2, ground_tags, name = "GroundPlane")
if patch_tags:
    pg_map["patch"] = gmsh.model.addPhysicalGroup(2, patch_tags, name = "Patch")
if port_tags:
    pg_map["lumped_port"] = gmsh.model.addPhysicalGroup(2, port_tags, name = "LumpedPort")
if farfield_tags:
    pg_map["farfield"] = gmsh.model.addPhysicalGroup(2, farfield_tags, name = "FarField")
if sides:
    pg_map["sides"] = gmsh.model.addPhysicalGroup(2, sides, name = "Sides")

### Refine and generation of the mesh

In [6]:
wavelength = 3e8 / (freq * 1e9)
mesh_size = wavelength / 15

refine_surfaces(port_tags + patch_tags, mesh_size) 

gmsh.model.mesh.generate(3)
gmsh.model.mesh.setOrder(1)

gmsh.option.setNumber("Mesh.MshFileVersion", 2.2)
gmsh.option.setNumber("Mesh.Binary", 0)
gmsh.write(filename)

gmsh.fltk.run()
gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 32 (Line)
Info    : [ 10%] Meshing curve 33 (Line)
Info    : [ 10%] Meshing curve 34 (Line)
Info    : [ 10%] Meshing curve 35 (Line)
Info    : [ 20%] Meshing curve 36 (Line)
Info    : [ 20%] Meshing curve 37 (Line)
Info    : [ 20%] Meshing curve 38 (Line)
Info    : [ 20%] Meshing curve 39 (Line)
Info    : [ 30%] Meshing curve 40 (Line)
Info    : [ 30%] Meshing curve 41 (Line)
Info    : [ 30%] Meshing curve 42 (Line)
Info    : [ 30%] Meshing curve 43 (Line)
Info    : [ 40%] Meshing curve 44 (Line)
Info    : [ 40%] Meshing curve 45 (Line)
Info    : [ 40%] Meshing curve 46 (Line)
Info    : [ 40%] Meshing curve 47 (Line)
Info    : [ 50%] Meshing curve 48 (Line)
Info    : [ 50%] Meshing curve 49 (Line)
Info    : [ 50%] Meshing curve 50 (Line)
Info    : [ 60%] Meshing curve 51 (Line)
Info    : [ 60%] Meshing curve 52 (Line)
Info    : [ 60%] Meshing curve 53 (Line)
Info    : [ 60%] Meshing curve 54 (Line)
Info    : [ 70%] Meshing curve 55

Info    : It. 500 - 401 nodes created - worst tet radius 1.79051 (nodes removed 0 99)
Info    : It. 1000 - 684 nodes created - worst tet radius 1.50238 (nodes removed 0 316)


Info    : It. 1500 - 974 nodes created - worst tet radius 1.34138 (nodes removed 0 526)
Info    : It. 2000 - 1248 nodes created - worst tet radius 1.22449 (nodes removed 0 752)
Info    : It. 2500 - 1574 nodes created - worst tet radius 1.13324 (nodes removed 0 926)
Info    : It. 3000 - 1903 nodes created - worst tet radius 1.07087 (nodes removed 0 1097)
Info    : It. 3500 - 2231 nodes created - worst tet radius 1.02198 (nodes removed 0 1269)
Info    : 3D refinement terminated (5799 nodes total):
Info    :  - 0 Delaunay cavities modified for star shapeness
Info    :  - 1346 nodes could not be inserted
Info    :  - 30102 tetrahedra created in 0.106421 sec. (282858 tets/s)
Info    : 6 node relocations
Info    : Done meshing 3D (Wall 0.216423s, CPU 0.211268s)
Info    : Optimizing mesh...
Info    : Optimizing volume 1
Info    : Optimization starts (volume = 4.68e-06) with worst = 0.0162924 / average = 0.432785:
Info    : 0.00 < quality < 0.10 :       286 elements
Info    : 0.10 < quality < 

-------------------------------------------------------
Version       : 4.15.2
License       : GNU General Public License
Build OS      : Linux64-sdk
Build date    : 20260324
Build host    : gmsh.info
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blas[petsc] Blossom Cgns DIntegration Dlopen DomHex Eigen[contrib] Fltk Gmm[contrib] Hxt Jpeg Kbipack Lapack[petsc] LinuxJoystick MathEx[contrib] Med Mesh Metis[contrib] Mmg Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP OptHom PETSc Parser Plugins Png Post QuadMeshingTools QuadTri Solver TetGen/BR TinyXML2[contrib] Untangle Voro++[contrib] WinslowUntangler Zlib tinyobjloader
FLTK version  : 1.3.11
PETSc version : 3.14.4 (real arithmtic)
OCC version   : 7.8.1
MED version   : 4.1.0
Packaged by   : geuzaine
Web site      : https://gmsh.info
Issue tracker : https://gitlab.onelab.info/gmsh/gmsh/issues
-------------------------------------------------------


### Generate JSON config


In [7]:
output_file: str = "sw_antenna.json"
freq_min: float = 3.0
freq_max: float = 3.5
freq_step: float = 0.005
eps_r: float = 2.2
loss_tan: float = 0.0009
port_impedance: float = 50.0
solver_order: int = 2

In [8]:
def attr(name):
        return [pg_map[name]] if name in pg_map else []

config = {
    "Problem": {
        "Type": "Driven",
        "Verbose": 2,
        "Output": "/work/results/sw_antenna/"
    },

    "Model": {
        "Mesh": f"/work/{filename}",
        "L0": 1.0,
        "Refinement": {}
    },

    "Domains": {
        "Materials": [
            {
                "Attributes": attr("substrate"),
                "Permittivity": eps_r,
                "Permeability": 1.0,
                "LossTan": loss_tan
            },
            {
                "Attributes": attr("air"),
                "Permittivity": 1.0,
                "Permeability": 1.0
            }
        ]
    },

    "Boundaries": {
        "PEC": {
            "Attributes": attr("ground_plane") + attr("patch")
        },

        "LumpedPort": [
            {
                "Index": 1,
                "Attributes": attr("lumped_port"),
                "R": port_impedance,
                "Excitation": True,
                "Direction": "+Z"
            }
        ],

        "Absorbing": {
            "Attributes": attr("farfield"),
            "Order": 1
        }
    },

    "Solver": {
        "Order": solver_order,
        "Device": "CPU",

        "Driven": {
            "MinFreq": freq_min,
            "MaxFreq": freq_max,
            "FreqStep": freq_step,
            "AdaptiveTol": 0.001
        },

        "Linear": {
            "Type": "Default",
            "KSPType": "GMRES",
            "Tol": 1.0e-8,
            "MaxIts": 200,
            "ComplexCoarseSolve": True
        }
    }
}

script_dir = os.getcwd()
config_path = os.path.join(script_dir, output_file)
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"Palace config written to {config_path}")

Palace config written to /home/martin/Desktop/PalaceToolkit/docs/examples/sw_antenna.json
